In [36]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 
TARGET = "Survived"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Experiment =[
    Name,
    Model_name,
    Model_params:{},
    feature_engineering:{},
    features:{numerical:{}, onehot:{}, ordinal:{}},
    preprocessing:{},
    evaluation:{},
    notes,
]

### Run_Experiments Blueprint - WIP:

run_experiments (Df, Experiments) -> Result_df for each experiment:

    Feature engineering function (DF, Experiments[feature_engineering]) -> this experiments modified_df:
        (creates/modifies columns)

    build_preprocessor(Experiment[[features, preprocessing,]]) -> preprocessor:
        (prepares selected columns for sklearn model)

    model(modified_df, preprocessor, Experiment[Model_name, Model_params, evaluation]) -> Prediction and evaluation:
        (trains/predicts/evaluates)

In [37]:
def Titanic_feature_engineering(df):

    df = df.copy()

    # Creating 'Deck' and 'Has_Cabin'
    df['Deck'] = df['Cabin'].str[0]
    df['Has_Cabin'] = df['Deck'].notnull().astype(int)

    # Creating 'Title'
    title_name = df['Name'].str.split(',').str[1]
    title = title_name.str.split('.').str[0]
    df['Title'] = title.str.strip()

    # Dropping 'PassengerId', 'Ticket', 'Cabin', 'Name'
    df.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1, inplace=True)

    # Creating 'Family_size', 'Alone', 'IsAgeMissing'
    df['Family_size'] = df['SibSp'] + df['Parch'] + 1
    df['Alone'] = (df['Family_size'] == 1).astype(int)
    df['IsAgeMissing'] = df['Age'].isna().astype(int)

    # Cutting Age to create Age_bin
    bins = [0, 14, 35, 60, 100]
    labels = ['0', '2', '3', '1']
    df['Age_bin'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

    # Creating 'Fare/Family_size', 'Fare/Age', 'Fare_bin'
    df['Fare/Family_size'] = df['Fare'] / df['Family_size']
    df['Fare/Age'] = df['Fare'] / df['Age']
    df['Fare_bin'] = pd.qcut(df['Fare'], q = 5, labels=['0','1', '2', '3', '4'])

    # Creating 'Pclass_Sex', 'Pclass_Title'
    df['Pclass_Sex'] = df['Pclass'].astype(str) + '_' + df['Sex'].astype(str)
    df['Pclass_Title'] = df['Pclass'].astype(str) + '_' + df['Title']

    # Filling Deck's Nan with 'None'
    df['Deck'] = df['Deck'].fillna('None')

    

    return df

In [38]:
from titanic_ml.common.experiments.runner import run_experiments
from titanic_ml.common.experiments import experiment_config

df = pd.read_csv(paths.TRAIN_PATH)
# print(df.head())

working_df = Titanic_feature_engineering(df)
print(working_df.head())

baseline = experiment_config.baseline
print(baseline)




   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked  Deck  \
0         0       3    male  22.0      1      0   7.2500        S  None   
1         1       1  female  38.0      1      0  71.2833        C     C   
2         1       3  female  26.0      0      0   7.9250        S  None   
3         1       1  female  35.0      1      0  53.1000        S     C   
4         0       3    male  35.0      0      0   8.0500        S  None   

   Has_Cabin Title  Family_size  Alone  IsAgeMissing Age_bin  \
0          0    Mr            2      0             0       2   
1          1   Mrs            2      0             0       3   
2          0  Miss            1      1             0       2   
3          1   Mrs            2      0             0       3   
4          0    Mr            1      1             0       3   

   Fare/Family_size  Fare/Age Fare_bin Pclass_Sex Pclass_Title  
0           3.62500  0.329545        0     3_male         3_Mr  
1          35.64165  1.875876     

In [39]:
def experiment_result_to_markdown(result_row):
    if hasattr(result_row, "to_dict"):
        result_row = result_row.to_dict()

    lines = [
        "| Field | Value |",
        "|---|---|",
    ]

    for key, value in result_row.items():
        lines.append(f"| {key} | {value} |")

    return "\n".join(lines)

In [40]:
def results_to_markdown(results_df):
    markdown = ""

    for index, row in results_df.iterrows():
        markdown += f"Experiment {row['experiment']}:\n"
        markdown += experiment_result_to_markdown(row) + "\n"

    return markdown

In [41]:
def experiment_result_to_markdown(result_row):
    if hasattr(result_row, "to_dict"):
        result_row = result_row.to_dict()

    lines = [
        f"Experiment {result_row.get('experiment', 'N/A')}:",
        "| Field | Value |",
        "|---|---|",
        f"|Train accuracy| {result_row.get('train_accuracy_mean', 'N/A')} ± {result_row.get('train_accuracy_std', 'N/A')} |",
        f"|Train precision| {result_row.get('train_precision_mean', 'N/A')} ± {result_row.get('train_precision_std', 'N/A')} |",
        f"|Train recall| {result_row.get('train_recall_mean', 'N/A')} ± {result_row.get('train_recall_std', 'N/A')} |",
        f"|Train f1| {result_row.get('train_f1_mean', 'N/A')} ± {result_row.get('train_f1_std', 'N/A')} |",
        f"|Test accuracy| {result_row.get('test_accuracy_mean', 'N/A')} ± {result_row.get('test_accuracy_std', 'N/A')} |",
        f"|Test precision| {result_row.get('test_precision_mean', 'N/A')} ± {result_row.get('test_precision_std', 'N/A')} |",
        f"|Test recall| {result_row.get('test_recall_mean', 'N/A')} ± {result_row.get('test_recall_std', 'N/A')} |",
        f"|Test f1| {result_row.get('test_f1_mean', 'N/A')} ± {result_row.get('test_f1_std', 'N/A')} |",
    ]
    return "\n".join(lines)

In [42]:
def format_metric(result_row, metric_name):
    mean_key = f"{metric_name}_mean"
    std_key = f"{metric_name}_std"

    mean = result_row.get(mean_key)
    std = result_row.get(std_key)

    if mean is None or std is None:
        return "N/A"

    return f"{mean} ± {std}"

In [43]:
def experiments_summary_to_markdown(results_df):
    summary_rows = []

    for _, row in results_df.iterrows():
        summary_rows.append({
            "experiment": row.get("experiment", "N/A"),
            "model_name": row.get("model_name", "N/A"),
            "status": row.get("status", "N/A"),
            "test_accuracy": format_metric(row, "accuracy"),
            "test_precision": format_metric(row, "precision"),
            "test_recall": format_metric(row, "recall"),
            "test_f1": format_metric(row, "f1"),
            "notes": row.get("notes", ""),
        })

    summary_df = pd.DataFrame(summary_rows)

    return summary_df.to_markdown(index=False)

In [44]:
import pprint

config_text = pprint.pformat(baseline, sort_dicts=False)
print(config_text)

[{'name': 'baseline_logreg',
  'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'],
  'feature_engineering': None,
  'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'],
                    'onehot_features': ['Sex', 'Embarked'],
                    'ordinal_features': ['Pclass'],
                    'numeric_imputer': 'median',
                    'categorical_imputer': 'most_frequent',
                    'scaler': 'standard'},
  'model_name': 'logreg',
  'model_params': {'max_iter': 1000, 'random_state': 42},
  'evaluation': {'method': 'cross_validation',
                 'cv': 5,
                 'scoring': ['accuracy', 'precision', 'recall', 'f1'],
                 'return_train_score': True,
                 'n_jobs': -1},
  'notes': 'Base logistic regression. Baseline for comparison.'}]


In [45]:
def experiment_result_to_markdown(result_row):
    if hasattr(result_row, "to_dict"):
        result_row = result_row.to_dict()

    lines = [
        f"Experiment {result_row.get('experiment', 'N/A')}:",
        "| Field | Value |",
        "|---|---|",
        f"|Train accuracy| {format_metric(result_row, 'train_accuracy')} |",
        f"|Train precision| {format_metric(result_row, 'train_precision')} |",
        f"|Train recall| {format_metric(result_row, 'train_recall')} |",
        f"|Train f1| {format_metric(result_row, 'train_f1')} |",
        f"|Test accuracy| {format_metric(result_row, 'test_accuracy')} |",
        f"|Test precision| {format_metric(result_row, 'test_precision')} |",
        f"|Test recall| {format_metric(result_row, 'test_recall')} |",
        f"|Test f1| {format_metric(result_row, 'test_f1')} |",
    ]
    return "\n".join(lines)

In [46]:
result = run_experiments(df, baseline,target=TARGET,)
print(result)
print(experiment_result_to_markdown(result.iloc[0]))

        experiment model_name   status error_type error_message  \
0  baseline_logreg     logreg  success                            

   test_accuracy_mean  test_accuracy_std  train_accuracy_mean  \
0               0.786              0.018                0.803   

   train_accuracy_std  test_precision_mean  ...  test_recall_std  \
0               0.005                0.736  ...            0.038   

   train_recall_mean  train_recall_std  test_f1_mean  test_f1_std  \
0              0.708             0.015         0.713        0.026   

   train_f1_mean  train_f1_std  fit_time_mean  score_time_mean  \
0          0.734         0.008          0.018            0.016   

                                               notes  
0  Base logistic regression. Baseline for compari...  

[1 rows x 24 columns]
Experiment baseline_logreg:
| Field | Value |
|---|---|
|Train accuracy| 0.803 ± 0.005 |
|Train precision| 0.762 ± 0.011 |
|Train recall| 0.708 ± 0.015 |
|Train f1| 0.734 ± 0.008 |
|Test accur

In [47]:
from titanic_ml.common.experiments.experiment_report import experiment_result_to_markdown, experiments_summary_to_markdown, experiment_config_to_markdown, experiment_report

exp_report = experiment_report(result, [baseline])

In [48]:
print(result)

        experiment model_name   status error_type error_message  \
0  baseline_logreg     logreg  success                            

   test_accuracy_mean  test_accuracy_std  train_accuracy_mean  \
0               0.786              0.018                0.803   

   train_accuracy_std  test_precision_mean  ...  test_recall_std  \
0               0.005                0.736  ...            0.038   

   train_recall_mean  train_recall_std  test_f1_mean  test_f1_std  \
0              0.708             0.015         0.713        0.026   

   train_f1_mean  train_f1_std  fit_time_mean  score_time_mean  \
0          0.734         0.008          0.018            0.016   

                                               notes  
0  Base logistic regression. Baseline for compari...  

[1 rows x 24 columns]
